# Extract.ipynb
This file will preprocess and extract neccessary info from `camma_mvor_2018.json` annotation file 

In [20]:
#importing neccessary packages and libraries
import json
from collections import defaultdict
import numpy as np
from general_utils import convertKeypoints, printKp, validateShape, printFrame, printOp 

In [21]:
#loading in pose annotations 
try:
    with open('camma_mvor_2018.json') as f:
        data = json.load(f)
except FileNotFoundError:
    print("file not found")

except:
    print("other error")

annot_3D = data["annotations3D"] #3D pose annotations


#ensuring annotations where loaded properly
print(annot_3D[0]['keypoints3D'])

[630.6556928399972, -498.0674644841253, 2394.9087204313873, 2.0, 650.6029327608261, -435.673747489747, 2467.1504194692443, 2.0, 765.2336487731467, -415.1274711986172, 2567.2710725550846, 2.0, 471.51789534403446, -417.4315698597386, 2576.9883790848066, 2.0, 762.8635121989911, -103.45690191919803, 2816.927522526169, 2.0, 493.88975427043096, -88.01155362794417, 2806.6194635029433, 2.0, 739.0484619140625, -224.3656005859375, 2560.0, 2.0, 334.1165332364915, -217.05363208174617, 2561.7882492948274, 2.0, 737.6575317382812, -122.52092742919922, 2450.0, 2.0, 478.4024783369531, -167.64844196983597, 2436.253111187903, 2.0, 787.6575317382812, -72.52092742919922, 2500.0, 1.0, 541.8294915212192, -182.77253333431509, 2388.574626849282, 2.0]


In [22]:
print(len(annot_3D)) #num keypoints

1061


## Validating keypoint shape

Looking at the first annotation entry, we see notice that first keypoint entry has more than 10 keypoints.


In [23]:
print(len(annot_3D[0]['keypoints3D']))

48


Given that each keypoint is of the form x,y,z,?  if there were 10 keypoints there should be the lenght of the entry should be 4x10=40. To verify if this is consistent across the whole dataset we will check keypoint count for every entry

In [24]:
lengths = [len(a['keypoints3D']) for a in annot_3D]
print(set(lengths))  # if one number, it's consistent

{48}


The code above proves a disrepency between, documentation and actual data shape. After further inspection, 4th coordinate seems to be irrelevant so it will be dropped from every frame. As for the the 11 and 12 keypoints respectively, after visualization they seem to represent keypoints for the hands, thus they will be left in.

## Reshaping annotations

48 value keypoint will now become a 12x3 list (12 keypoints , xyz coordinates)

In [25]:
#reshaping all keypoints
for annot in annot_3D:
    annot['keypoints3D']=convertKeypoints(annot['keypoints3D'])




In [26]:
printKp(annot_3D[0]['keypoints3D'])

1: [630.6556928399972, -498.0674644841253, 2394.9087204313873]
2: [650.6029327608261, -435.673747489747, 2467.1504194692443]
3: [765.2336487731467, -415.1274711986172, 2567.2710725550846]
4: [471.51789534403446, -417.4315698597386, 2576.9883790848066]
5: [762.8635121989911, -103.45690191919803, 2816.927522526169]
6: [493.88975427043096, -88.01155362794417, 2806.6194635029433]
7: [739.0484619140625, -224.3656005859375, 2560.0]
8: [334.1165332364915, -217.05363208174617, 2561.7882492948274]
9: [737.6575317382812, -122.52092742919922, 2450.0]
10: [478.4024783369531, -167.64844196983597, 2436.253111187903]
11: [787.6575317382812, -72.52092742919922, 2500.0]
12: [541.8294915212192, -182.77253333431509, 2388.574626849282]


## Frame by frame pose grouping 
Relevant information to extraction from each annotation
- `image_ids`
- `person_id`
- `keypoints3D`

Poses per frame will be stored in dictionary with the following shape: 

```python
    poses_per_frame = {
        frame_1: [ {pose1}, ... , {poseN}],
        ...
        frame_N: [ {pose1}, ... , {poseN}],
    }
```



In [27]:
poses_per_frame = defaultdict(list) 

for annot in annot_3D:
    frame = annot['image_ids']
    person = annot['person_id']
    keypoints = annot['keypoints3D']


    poses_per_frame[frame].append({
        'person': person,
        'keypoints': keypoints
    })

#inspect data to make sure it was parsed properly 
print(f"There's {len(poses_per_frame)} total frames") 
print(f"Frame 1: {list(poses_per_frame.values())[0]}")

multiPersonFrame = 0
for pose in list(poses_per_frame.values()):
    if len(pose)>1:
        multiPersonFrame +=1

print(f"Theres {multiPersonFrame} frames with more than one person")

There's 629 total frames
Frame 1: [{'person': 0, 'keypoints': [[630.6556928399972, -498.0674644841253, 2394.9087204313873], [650.6029327608261, -435.673747489747, 2467.1504194692443], [765.2336487731467, -415.1274711986172, 2567.2710725550846], [471.51789534403446, -417.4315698597386, 2576.9883790848066], [762.8635121989911, -103.45690191919803, 2816.927522526169], [493.88975427043096, -88.01155362794417, 2806.6194635029433], [739.0484619140625, -224.3656005859375, 2560.0], [334.1165332364915, -217.05363208174617, 2561.7882492948274], [737.6575317382812, -122.52092742919922, 2450.0], [478.4024783369531, -167.64844196983597, 2436.253111187903], [787.6575317382812, -72.52092742919922, 2500.0], [541.8294915212192, -182.77253333431509, 2388.574626849282]]}]
Theres 313 frames with more than one person


In [28]:
for key in poses_per_frame.keys():
    print(key)

30010000128_30020000128_30030000128
20010000031_20020000031_20030000031
30010000030_30020000030_30030000030
40010000060_40020000060_40030000060
20010000297_20020000297_20030000297
40010000053_40020000053_40030000053
20010000177_20020000177_20030000177
20010000194_20020000194_20030000194
20010000270_20020000270_20030000270
20010000169_20020000169_20030000169
20010000005_20020000005_20030000005
20010000101_20020000101_20030000101
20010000228_20020000228_20030000228
20010000151_20020000151_20030000151
30010000177_30020000177_30030000177
40010000022_40020000022_40030000022
30010000113_30020000113_30030000113
30010000165_30020000165_30030000165
20010000134_20020000134_20030000134
10010000056_10020000056_10030000056
40010000020_40020000020_40030000020
40010000104_40020000104_40030000104
40010000030_40020000030_40030000030
40010000106_40020000106_40030000106
20010000079_20020000079_20030000079
20010000163_20020000163_20030000163
20010000278_20020000278_20030000278
20010000184_20020000184_2003

In [29]:
list(poses_per_frame.values())[0:2]

[[{'person': 0,
   'keypoints': [[630.6556928399972, -498.0674644841253, 2394.9087204313873],
    [650.6029327608261, -435.673747489747, 2467.1504194692443],
    [765.2336487731467, -415.1274711986172, 2567.2710725550846],
    [471.51789534403446, -417.4315698597386, 2576.9883790848066],
    [762.8635121989911, -103.45690191919803, 2816.927522526169],
    [493.88975427043096, -88.01155362794417, 2806.6194635029433],
    [739.0484619140625, -224.3656005859375, 2560.0],
    [334.1165332364915, -217.05363208174617, 2561.7882492948274],
    [737.6575317382812, -122.52092742919922, 2450.0],
    [478.4024783369531, -167.64844196983597, 2436.253111187903],
    [787.6575317382812, -72.52092742919922, 2500.0],
    [541.8294915212192, -182.77253333431509, 2388.574626849282]]}],
 [{'person': 0,
   'keypoints': [[528.8969469954174, -656.7317272990944, 2534.672794919807],
    [410.12113632656565, -640.9080534956912, 2583.1776669826204],
    [447.07905692298135, -689.0041633356487, 2780.021906403506

In [30]:
#checking frame distribution
personFreq = {}

for frame in list(poses_per_frame.values()):
    numPoses = len(frame)
    if numPoses not in personFreq:
        personFreq[numPoses] = 0
    else: 
        personFreq[numPoses] +=1

print(personFreq)

{1: 315, 2: 222, 3: 68, 4: 14, 7: 0, 5: 4}


In [31]:
#checking how many pose annotations in total
total = 0
for pose in list(poses_per_frame.values()):
    total += len(pose)

print(total)

1061


## Per operation grouping
Based on pose grouping above, frames will now be grouped per operation. Operations will be stored in a dictionary with the following shape:

```python
    operations = {
        day_x: {operation_x: [frames]}

    }
```

In [32]:
OP_RANGES = { #frame span of operations
    'day1': [
        (0, 56)
    ],
    'day2': [
        (0, 117),
        (118, 232),
        (233, 234),
        (235, 245),
        (246, 257),
        (258, 270),
        (271, 278),
        (279, 293),
        (294, 321),
        (322, 324), 
        (325, 329)
    ],
    'day3': [
        (0, 7),
        (8, 28),
        (29, 58),
        (59, 84),
        (85, 124),
        (125, 138),
        (139, 160),
        (161, 172),
        (173, 181),
        (182, 196),
        (197, 217)
        # ommited 3 imgs at end of day 3  because they didnt constitute a full operation
    ],
    'day4': [
        (0, 75),
        (76, 116),
        (117, 121)
    ]
}

operations = defaultdict(lambda: defaultdict(list)) 
def getIndex(imageId): #extracts image index from id 
    return int(imageId.split('_')[0][-6:]) #ex: 10010000002_10020000002_10030000002 --> 000002 --> 2)

def getDay(imageId): #extracts day from id
    day = int(imageId.split('_')[0][0])

    match day:
        case 1:
            return 'day1'
        
        case 2:
            return 'day2'
        
        case 3:
            return 'day3'
        
        case 4:
            return 'day4'
    

#iterating through frames and mapping them to their respective operations 
for frameId, poses in poses_per_frame.items():
    dayString = getDay(frameId)
    frameIdx = getIndex(frameId)


    for opIdx, (opStart, opEnd) in enumerate(OP_RANGES[dayString], start =1):
        
        if opStart <= frameIdx <= opEnd:
            opString = f"operation_{opIdx}"
            operations[dayString][opString].append({
                'frameIdx': frameIdx,
                'poses': poses 
            })
            break #make sure it only maps to one operation

#sort to put frames in order
for day in operations:
    for op in operations[day]:
        operations[day][op].sort(key=lambda f: f['frameIdx'])



### Inspecting data 

In [33]:
#Visualizing operation annotations
print(f"Day 1, operation 1: {operations['day1']['operation_1']}")

Day 1, operation 1: [{'frameIdx': 1, 'poses': [{'person': 0, 'keypoints': [[-919.0739482366955, 220.82523959568664, 1500.285433128346], [-900.6188055769591, 269.0215963454077, 1591.8862113213054], [-767.3401648708726, 254.80329475774147, 1701.0095984835384], [-1062.8579020560446, 242.84780550162645, 1779.6725563248588], [-707.1937682188919, 473.3078621762493, 1983.514584502264], [-971.708986329872, 629.1873350132807, 1986.7901324341426], [-642.777364167334, 357.5466914290271, 1908.2174367030243], [-1093.1801297857166, 456.7583338835688, 1950.6948686629373], [-602.3968193964831, 574.8271083699042, 1919.106531333714], [-1117.1871539088602, 665.6120719028361, 2062.1952132462443], [-596.7772216796875, 643.4946899414062, 1883.0], [-1154.817825383358, 691.7864954734459, 2068.0941845246466]]}]}, {'frameIdx': 3, 'poses': [{'person': 1, 'keypoints': [[-227.58989221610074, -184.0425039183756, 2313.0511787435516], [-233.268458343472, -170.6410320503951, 2410.820223440352], [-68.43472747519547, -6

In [34]:
#inspecting total number of operations 
total = 0
for day in operations:
    for op in operations[day]:
        for frame in operations[day][op]:
            total += len(frame['poses'])

print(total)

1058


In [35]:
#Seeing how many operations have under 10 annotated frames 

under10 = 0
total = 0

for day in operations:
    for op in operations[day]:
        total+=1
        if len(operations[day][op])<10:
            under10+=1

print(f"{under10}/{total} operations have less than 10 frames")
    

9/26 operations have less than 10 frames


## Building frame vector

shape: `[Mean person config  | frame diversity | Scene  graph summary ]`

In [36]:
## exporting per operation grouping to work with cleaner data

with open('poses_per_operation.json', 'w') as f:
    json.dump(operations, f)

In [37]:
#importing new json
try:
    with open('poses_per_operation.json') as f:
        cleanedAnnots = json.load(f) #cleaned annotations
except FileNotFoundError:
    print("file not found")

except:
    print("other error")

In [38]:
## converting keypoint arrays into numpy arrays
for day in cleanedAnnots:
    for operation in cleanedAnnots[day]:
        for frame in cleanedAnnots[day][operation]:
            for pose in frame['poses']:
                pose['keypoints'] = np.array(pose['keypoints'])



In [39]:
#testing validity of keypoint structure
validateShape(cleanedAnnots)

All keypoints valid: numpy arrays of shape (12, 3)


In [40]:
##exporting cleaned annotations so it can be worked on in gpd notebook
import pickle 

with open('cleanedAnnots.pkl', 'wb') as f:
    pickle.dump(cleanedAnnots, f)
    